In [1]:
print("hello world")

hello world


In [2]:
import requests
import json
from astropy.time import Time
from datetime import datetime
import numpy as np
import selecting_3_obs as s3
import gauss as g
import random

In [3]:
# Query observations for Bennu in OBS80 format
def get_asteroid_obs(asteroids_desig):
    response = requests.get(
        "https://data.minorplanetcenter.net/api/get-obs",
        json={"desigs": [asteroids_desig], "output_format": ["OBS80"]}
    ) 
    if response.ok:
        return response
    else:
        print(f"Error: {response.status_code} - {response.content}")
        raise ValueError
    

rr = get_asteroid_obs("Bennu")


In [4]:
def number_check(num):
    """provided a string, check all characters on the string and removes  all the characters that are not a dot or a number

    Args:
        num (string): a string that should be a number (or want to make a number)

    Returns:
        string: a string that can be comverted to a float
    """
    try:
            int(num)
    except:
        for el in num:
            if el.isdigit() or el == '.':
                continue
            else:
                num = num.replace(el,"")
    return num

def date_1(line):
    """gets the original date from a line of a obs type file
    Args:
        line (str): a string with the format of .obs

    Returns:
        str: the date included in the line
    """
    line = line[15:]
    elements = line.split('.')
    to_add = elements[1].split(' ')
    time = elements[0]
    time += '.' + to_add[0]
    return time

def format_date(date):
    """format the date that is ouputed from function date_1 in the format year_month_day(as a fraction) into more specific elements (includes seconds) 

    Args:
        date (str): the original date in the formate previuly indicated year_month_day

    Returns:
        date: a julian date
    """
    
    year = ''
    initial = date.split()
    for i in range(len(initial)):
        initial[i] = number_check(initial[i])

    year = (initial[0][-4]+ initial[0][-3]+ initial[0][-2]+initial[0][-1])
    year = int(year)
    mon = int(initial[1])
    og_day = initial[2]
    days = initial[2].split('.')
    day = int(days[0])  
    h = str(float('0.'+days[1]) * 24).split('.')
    hor = int(h[0])
    m = str(float('0.' + h[1]) * 60).split('.')
    min = int(m[0])
    s = str(float('0.' + m[1]) * 60).split('.')
    if len(s) != 2 and "." not in str(float('0.' + m[1]) * 60):
        # checks that the seconds are greater than not zero and if they are adds make them zero and calculates the microseconds
        s = [0,s[0]]
        sec = int(s[0])
        microsec = int(float(s[1]) * 1000000)
    else:
        sec = int(s[0])
        microsec = int(float('0.' + s[1]) * 1000000)
    date = datetime(year,mon,day,hor,min,sec,microsec)
    # converting date to julian date 
    t = Time(date)
    julian_date = t.jd
    return t.jd

def obs_code_1(line):
    """Gets observatory code of a line with .obs format
    Args:
        line (str): a line with .obs format such as: 'F3814K01W05N  C1996 02 10.39593 09 32 55.96 +12 14 44.3          21.4 Vah8170691'
    Returns:
        str: The observatory code in the lines provided (the last three digits of the line)
    """
    code = line[-3:]
    return (code)

In [5]:
def read_api_response_obs(response):
    text_results = response.json()[0]["OBS80"]
    lines = text_results.strip().split('\n')
    result = []
    ra_dec = []
    for ln in lines:
        x = ln.split(" ")
        while "" in x:
            x.remove("")
        for i in range(len(x)):
            if "+" in x[i] and len(x[i]) >4:
                x[i:i+1] = x[i].split("+")
            if "-" in x[i] and len(x[i]) >4:
                x[i:i+1] = x[i].split("-")
        if len(x[3]) >= 10:
            x[3:4] = [x[3][:-2], x[3][-2:]] 
            try:
                np.array(x[4:10],dtype=float)
            except:
                print("There is an error in getting the data: ", x[4:10])
                print(ln)
        ra_dec.append(x[4:10])
        result.append(np.array([format_date(date_1(ln)), obs_code_1(ln)]))
    result = np.array(result)
    dates_obs = np.array(result[:,0],dtype=float)
    obs_codes = np.array(result[:,1])
    ra_dec=np.array(ra_dec)
    assert len(ra_dec) == len(obs_codes) == len(dates_obs)
    return dates_obs, obs_codes, ra_dec

all_dates,all_codes, all_righta_declin = read_api_response_obs(rr)

In [28]:
len(all_dates)

603

In [29]:
obs_combo = s3.create_A_3obs_sets(all_dates)

In [31]:
len(obs_combo)

1511

In [7]:
def chose_obs_sets(obs_sets: np.array,num_sets=10,indexes=[]):
    """provide with a list of multiple sets of 3 observations randomly and 
    symetrically chooses the specifide number of sets. 

    Args:
        obs_sets (np.array): contains all sets of 3 observations to choose from
        num_sets (int, optional): How many set you want it to return. Defaults to 10.
        indexes (list, optional): if you know the indexes of the sets that you want to
        choose you can manually provide them. Defaults to [].

    Raises:
        ValueError: when provided indexes is not a valid number (index out of range or not a number)

    Returns:
        np.array: radomly selcted sets from the provided list of sets. 
    """
    if not indexes:
        # select at random, from similarly spaced spaces/sets, (subspace)
        total_sets = len(obs_sets)
        subspace_size = total_sets // num_sets # decide normal size of a subspace
        remainder = total_sets % num_sets # number of additions that will be done to first couple of subspaces
        # 12 element per subspace
        # 4 reminder
        # Making subspace with the same as the number of sets
        subspaces = []
        start = 0 # initiation first subspace on the first index
        for i in range(num_sets):
            # Adjust subspace size for remainder
            j = 0 # for keeping track of if one (index) should be added to the
            # current subspace (if there is a reminder)
            if remainder != 0:
                j = 1
                remainder -=1
            end = start + subspace_size + j
            subspaces.append(obs_sets[start:end])
            start = end
        # select a random values from each of the subspaces
        selected_values = []
        for quad in subspaces:
            value = random.choice(quad)
            # inn = np.where(obs_sets == value)[0]
            selected_values.append(value)
        selected_values = np.array(selected_values)
        
    else:
        # When indexes are directly provided by the user, those indesex are choosen to retun the sets of dates. 
        try:
            selected_values = np.array(obs_sets[indexes])
        except:
            print("Please provide valid indexes for choosing sets of dates")
            print("selecting_3_obs.py, chose_obs_sets")
            raise ValueError
    return selected_values

In [8]:
dates_sets = chose_obs_sets(obs_combo,100)

In [32]:
dates_sets

array([[2453623.50277     , 2453626.08139     , 2453628.23537     ],
       [2453623.50277     , 2453626.09223     , 2453630.9274399998],
       [2453623.50277     , 2453626.13331     , 2453628.24997     ],
       [2453623.50277     , 2453626.89543     , 2453630.9274399998],
       [2453623.50277     , 2453628.23201     , 2453630.95472     ],
       [2453623.50277     , 2453628.23851     , 2453631.45585     ],
       [2453623.50277     , 2453628.24997     , 2453630.89237     ],
       [2453623.50277     , 2453628.24997     , 2453630.95472     ],
       [2453623.50378     , 2453626.08139     , 2453628.24829     ],
       [2453623.50378     , 2453626.13331     , 2453628.23851     ],
       [2453623.50378     , 2453626.87748     , 2453631.45543     ],
       [2453623.50378     , 2453628.23201     , 2453631.45543     ],
       [2453623.50378     , 2453628.23453     , 2453630.89237     ],
       [2453623.50378     , 2453628.2362      , 2453631.45585     ],
       [2453623.50378     , 245362

In [9]:
def connect_set_indeces(all_dates: list, three_sets: list):
    dates_float = all_dates.astype(float) # gets list of dates all dates as floats
    indexes_sort = np.argsort(dates_float)  # sorts the dates, (just in case it is not sorted)
    dates_sorted = dates_float[indexes_sort] # organizes the dates back, with the sorted indexes 
    index_found = np.searchsorted(dates_sorted,three_sets) # finds the index of the dates sorted (to get the obs code) in the flattened sets of 3 observations choosen
    indices = indexes_sort[index_found] # getting a list of indexes based on the sorted search and found indexes.
    # indices = indices.reshape(sets.shape) #  reshaping the gather indeces to be iin our desired format
    return indices

def collect_index(all_codes: list,all_righta_declin: list,indices: np.array):
    code = np.array(all_codes[indices]) # getting the indeices of the obs codes from json data
    ra_dec = np.array(all_righta_declin[indices])
    return code, ra_dec

index = connect_set_indeces(all_dates,dates_sets)
codes_sets , ra_dec_sets= collect_index(all_codes,all_righta_declin,index)

In [35]:
ra_dec_sets[0]

array([['02', '43', '14.88', '-13', '29', '59.4'],
       ['03', '12', '19.17', '-09', '11', '06.5'],
       ['03', '43', '10.41', '-04', '17', '11.4']], dtype='<U6')

In [10]:
def ra_to_radians(h,m,s):
    return np.deg2rad((h + m/60 + s/3600)*15)
    
def dec_to_radians(d,arcm,arcs):
    return np.deg2rad( arcm/60 + arcs/3600 + d)

In [11]:
# Right ascencion Obs format
# HH MM SS.SS
# hours minutes seconds

# Declination given format:
# ±DD MM SS.S
try: 
    ra_hours = g.gauss_method.flat_variable(np.array(ra_dec_sets[:,:,0],dtype=float))
    ra_minutes = g.gauss_method.flat_variable(np.array(ra_dec_sets[:,:,1],dtype=float))
    ra_seconds = g.gauss_method.flat_variable(np.array(ra_dec_sets[:,:,2],dtype=float))
    dec_degrees = g.gauss_method.flat_variable(np.array(ra_dec_sets[:,:,3],dtype=float))
    dec_arcminutes = g.gauss_method.flat_variable(np.array(ra_dec_sets[:,:,4],dtype=float))
    dec_arcseconds = g.gauss_method.flat_variable(np.array(ra_dec_sets[:,:,5],dtype=float))
except:
    print("The data type of at least one values is incorrect please check the data.")
    print()
    raise TypeError 


In [12]:
# Right ascencion Obs format
# HH MM SS.SS
# hours minutes seconds

# Declination given format:
# ±DD MM SS.S
# degrees arcminutes arcseconds

#progamr uses both RA and DEC in radians 

def ra_to_radians(h,m,s):
    return np.deg2rad((h + m/60 + s/3600)*15)
    
def dec_to_radians(d,arcm,arcs):
    return np.deg2rad( arcm/60 + arcs/3600 + d)


In [13]:
ra_radians = ra_to_radians(ra_hours,ra_minutes,ra_seconds)
dec_radians = dec_to_radians(dec_degrees,dec_arcminutes,dec_arcseconds)

In [14]:
ra_radians = ra_radians.reshape(dates_sets.shape)
dec_radians = dec_radians.reshape(dates_sets.shape)

In [37]:
len(dec_radians)

100

In [15]:
unit_vectors = g.gauss_method.get_unit_matrix(ra_radians,dec_radians)

In [ ]:
# from wis.wis import Wis# from wis.wis import wis

# import wis 

In [ ]:
def position_obs_wis(mjd,obscode): # uses Wis.py
    """Uses Wis library to calculate the position of the observatory in a Heliocentric equatorial frame. 
    Provided the mjd and observatory code. the mjd can be provided as sets of dates, however the 
    observatory code can only be a single string

    Args:
        mjd (float or list): list of dates of the observation 
        obscode (string): Observatory code of the observation

    Returns:
        np.matrix: a 3x3 matrix per each of the given times 
        (such as provided a list of 4 times, it returns a list with 4, 3x3 matrices corresponding to each time) Each retunr line corresponds to a single time
    """
    # flattening the parameters 
    jd_times = g.gauss_method.flat_variable(mjd)
    obscode = g.gauss_method.flat_variable(obscode)
    # jd_times = gm.mjd_to_jd(mjd)
    
    if type(obscode) == str or type(obscode) == int or len(obscode) == 1:
        obscode = np.repeat(obscode,len(mjd))
    
    if len(obscode) != len(jd_times):
        raise ValueError("Please provide the same number of observatory codes and times")
    # grouping observatory codes and corresponding times in a dictonary
    dict_code_times = {}
    for i in range(len(obscode)):
        if obscode[i] not in dict_code_times:
            dict_code_times[obscode[i]] = []
        dict_code_times[obscode[i]].append([jd_times[i],i])  # putting index on the dictonary for reodering 
   
    positions = np.zeros((len(mjd),3))
    # with wis.wis() as W:
    for code, values in dict_code_times.items():
        values = np.array(values)
        jd_times = values[:,0]
        indexes = np.array(values[:,1],dtype=int)
        times = Time(jd_times, format='jd', scale='tdb')
        # here use W object 
        #     with Wis() as W:
        # W.get_obs_helio_equ_AU(obscode,times) function will tell you position of observatory
        W = wis.wis(str(code), times) # as w and pass the w
        try:
            observer_posns = W.hXYZ
        except:
            print("please check, code and dates.")
            print(code)
            print(values) # provides dates, and index of the date 
            continue
        observer_posns = np.array(observer_posns)
        positions[indexes] = observer_posns
    return positions

In [18]:
flat_codes =  g.gauss_method.flat_variable(codes_sets)
flat_dates =  g.gauss_method.flat_variable(dates_sets)

In [19]:

position_vectors = position_obs_wis(flat_dates,flat_codes)

wis.py, Ground ... 
wis.py, Ground ... 
wis.py, Ground ... 
wis.py, Ground ... 
wis.py, Ground ... 
wis.py, Ground ... 
wis.py, Ground ... 
wis.py, Ground ... 
wis.py, Ground ... 
wis.py, Ground ... 
wis.py, Ground ... 


In [20]:
n = len(position_vectors) // 3
position_vectors = position_vectors.reshape(n, 3, 3)


In [21]:
observatory_position_vectors = position_vectors.transpose(0, 2, 1)

In [22]:
def jd_to_mjd(dates):
    date = g.gauss_method.flat_variable(dates)
    date = date - 2400000.5
    return  date

In [23]:
complete_dates = jd_to_mjd(dates_sets)

In [24]:
complete_dates = complete_dates.reshape((int(len(complete_dates)/3),3))

In [25]:
complete_dates.shape

(100, 3)

In [ ]:

try1 = g.gauss_method.gauss_method(complete_dates,unit_vectors,observatory_position_vectors)
print(len(try1[0]))
print(try1[0])

100
[ True  True  True  True  True  True  True  True  True  True False False
  True  True  True False False  True False False  True False False  True
  True  True  True  True  True  True  True False False False False  True
  True  True False False False  True False False  True  True False False
  True False False  True False False  True False False  True False False
  True False False  True False False  True False False  True False False
  True False False  True False False  True False False  True False False
  True  True False  True  True False  True  True False  True  True False
  True False False  True  True False  True  True False  True False False
  True False False  True  True False  True False False  True False False
  True False False  True False False  True False False  True  True False
 False  True  True False False  True False False  True False  True  True
 False  True  True False  True  True False  True  True False  True  True
 False  True  True False  True  True False  Tru

In [38]:
print(try1[0])

[[1.0051829886332624e+00 1.0000000000000000e-04 1.0000000000000000e-04]
 [1.0049965352769310e+00 1.0000000000000000e-04 1.0000000000000000e-04]
 [1.0051087815454078e+00 1.0000000000000000e-04 1.0000000000000000e-04]
 [1.0061865524176739e+00 1.0000000000000000e-04 1.0000000000000000e-04]
 [3.0314242296000482e+00 1.1556492698212237e+00 1.0058756922747611e+00]
 [1.3932418330852734e+00 1.0199541214532071e+00 8.9772848928507543e-01]
 [1.3879611435135537e+00 1.0167550152396578e+00 8.9838180659329381e-01]
 [1.1083913931840546e+00 1.0245175227801728e+00 9.2905816384989603e-01]
 [1.0051651591918078e+00 1.0000000000000000e-04 1.0000000000000000e-04]
 [1.0051156686142504e+00 1.0000000000000000e-04 1.0000000000000000e-04]
 [1.0064967857289249e+00 1.0000000000000000e-04 1.0000000000000000e-04]
 [2.4563814773906647e+00 1.0110237715641306e+00 9.1749677315627531e-01]
 [1.9735869491443199e+00 1.3255925740548549e+00 1.0072746718347434e+00]
 [1.6660780221053539e+00 1.0144971215306795e+00 8.98405380764727

In [47]:
import csv

In [ ]:
import csv


data = [
    [1, "a", 3.14],
    [2, "b", 2.71],
]

with open("data.csv", "w", newline="", buffering=1_000_000) as f:
    writer = csv.writer(f)
    writer.writerows(data)


In [ ]:
response = requests.get(
    "https://data.minorplanetcenter.net/api/get-orb",
    json={"desig": "Bennu"}
)
# careful about epoch with car, otherwise use kep

if response.ok:
    result = response.json()
    mpc_orb = result[0]['mpc_orb'][0]
    print(json.dumps(mpc_orb, indent=4))
else:
    print(f"Error: {response.status_code}")
    print(response.content)

{
    "CAR": {
        "coefficient_names": [
            "x",
            "y",
            "z",
            "vx",
            "vy",
            "vz"
        ],
        "coefficient_uncertainties": [
            6.78567e-09,
            3.64285e-08,
            6.30474e-09,
            3.44433e-10,
            2.4088e-11,
            3.2268e-10
        ],
        "coefficient_values": [
            -1.18999120399445,
            -0.251459363996825,
            -0.0222311251789383,
            0.000300061449107021,
            -0.0148709869455561,
            -0.00157179902599519
        ],
        "covariance": {
            "cov00": 4.604530318761083e-17,
            "cov01": -2.211963924941804e-16,
            "cov02": -1.015507322993966e-17,
            "cov03": 1.96849609469206e-18,
            "cov04": -4.378326861831438e-20,
            "cov05": 8.68784017166878e-19,
            "cov06": null,
            "cov07": null,
            "cov08": null,
            "cov09": null,
      